In [ ]:
import pygame
import time
import os
import tkinter as tk
from tkinter import *
import random
import glob
import pathlib


In [ ]:
season = "Sunny" #TODO User input? Random? Matching current season? 

current_time = time.asctime()
hour = current_time.split()[3].split(":")[0]
minute = current_time.split()[3].split(":")[1]

pygame.mixer.init(frequency=16000)
pygame.init()
path_music = os.getcwd()


In [ ]:
# Create a dictionary associating seasons and sound with a white noise
white_noises = {
    # Outside noises
    "Sunny": ["Steps_grass.mp3", "Running_grass.mp3", "Silence.mp3"],
    "Rainy": ["Rain_umbrella.mp3", "Rain_rooftop.mp3", "Silence.mp3"],
    "Autumn": ["Steps_leaves.mp3", "Rain_umbrella.mp3", "Silence.mp3"],
    "Snowy": ["Steps_snow.mp3", "Steps_woods.mp3", "Silence.mp3", "Fireplace.mp3"],

    # Inside noises
    "Museum": ["Steps_marble.mp3"],
    "Office": ["Office_sounds.mp3"]
}

In [ ]:
from mutagen.mp3 import MP3

path = os.path.join(path_music, "acnr-soundtrack","Resident services", f"*.mp3")
files = glob.glob(path)
i = 0

for mp3_file in files:
   audio = MP3(mp3_file)
   sample_freq = audio.info.sample_rate
   print(f"File {i}: frequenza di campionamento: {sample_freq} Hz") 
   i = i + 1

In [ ]:
path = os.path.join(path_music, "acnr-soundtrack","Resident services", f"*.mp3")
songs_list = glob.glob(path)
songs_list


# Version 2: No functions

In [ ]:
def choose_music():
    global ambience_old
    global noise_channel
    
    if random.random() > 0.5: # Hourly music, outside noises
        ambient = "Outdoor"
        ambience_new = "out"
        current_time = time.asctime()
        hour = current_time.split()[3].split(":")[0]
        path = os.path.join(path_music, "acnr-soundtrack", season, f"{hour}_{season}*mp3")
        song = glob.glob(path)[0]
    
        # Environmental sounds
        # Select the "Folder" according to the season
        sounds_list = white_noises[season]
        sound_name = random.choice(sounds_list)
        sound = pygame.mixer.Sound(os.path.join(path_music, "acnr-soundtrack","White noises", sound_name))
        
    else: # Set outside noises
        ambience_new = "in"
        path = os.path.join(path_music, "acnr-soundtrack","Resident services", f"*.mp3")
        songs_list = glob.glob(path)
        song = random.choice(songs_list)
    
        if "Museum" in song:
            ambient = "Museum"
        if "Office" in song:
            ambient = "Resident Services"
        else:
            if "Blather" in song:
                ambient = "Blather's tent"
            if "Airport" in song:
                ambient = "Airport"
            if "Able" in song:
                ambient = "Able's sisters"
            if "Nook" in song:
                ambient = "Nook's cranny"
            if "Jolly" in song:
                ambient = "Volpolo"
            if "Dreaming" in song:
                ambient = "Dreaming"
    
        match ambient:
            case "Museum":
                sound_name = random.choice(white_noises["Museum"])
            case "Resident services":
                sound_name = random.choice(white_noises["Office"])
            case _: # Other settings with no background sound
                sound_name = "Silence.mp3"
            
        sound = pygame.mixer.Sound(os.path.join(path_music, "acnr-soundtrack","White noises", sound_name))

    global music_label
    music_label.config(text= (pathlib.Path(song).stem).replace("_", ".").split(".", 1)[1])
    music_label.pack()

    # Play the music
    pygame.mixer.music.load(song)
    pygame.mixer.music.play()
     # Play background sounds
    if ambience_new != ambience_old:
        noise_channel.fadeout(1000)
        noise_channel.play(sound, loops=-1, fade_ms=2000)    
    

In [ ]:
pygame.mixer.init(frequency=44100, size = -16, channels = 2, buffer = 1024)
pygame.init()
path_music = os.getcwd()
noise_channel = pygame.mixer.Channel(1) # Create a channel dedicated to background sounds
ambience_old = "None"
season = 'Sunny'

window = tk.Tk()
window.geometry("500x300")
window.title("Radio Tom Nook")
window.resizable(False, False)
music_label = Label(window, text = "")

# Show the time
time_label = Label(window, text ="")
hour_show = time.asctime().split()[3].split(":")[0]
minute_old = time.asctime().split()[3].split(":")[1]
time_label.config(text = f"{hour_show}:{minute_old}")
time_label.pack()
def update_time():
    hour_show = time.asctime().split()[3].split(":")[0]
    minute_new = time.asctime().split()[3].split(":")[1]

    global minute_old
    if minute_new != minute_old:
        time_label.config(text = f"{hour_show}:{minute_new}")
        minute_old =  minute_new
    
    time_label.after(1000, update_time)
update_time()

# Choose season. Order: Sunny, Rainy, Snowy
def change_season():
    global season
    if season == "Sunny":
        season = "Rainy"
        season_button['text'] = ("Rainy")
    elif season == "Rainy":
        season_button['text'] = ("Snowy")
        season = "Snowy"
    elif season == "Snowy":
        season_button['text'] = ("Sunny")
        season = "Sunny"

season_button = tk.Button(text = "Sunny", command = change_season)
season_button.place(relx = 0.8, rely = 0.1)

choose_music()
pygame.mixer.music.pause()
noise_channel.pause()

is_paused = True

def handle_music():
    if pygame.mixer.music.get_busy():
        pygame.mixer.music.pause()
        noise_channel.pause()
        pause_button['text'] = ("Play!")

    else:
        pygame.mixer.music.unpause()
        noise_channel.unpause()
        pause_button['text'] = ("Pause!")

pause_button = tk.Button(text="Play!", command = handle_music)
pause_button.place(relx = 0.5, rely = 0.85)

# Set volume: music
pygame.mixer.music.set_volume(0.2)
up_music_button = tk.Button(text = "Up", command = lambda: pygame.mixer.music.set_volume(pygame.mixer.music.get_volume() + 0.1))
down_music_button = tk.Button(text = "Down", command = lambda: pygame.mixer.music.set_volume(pygame.mixer.music.get_volume() - 0.1))
up_music_button.place(relx = 0.1, rely = 0.1)
down_music_button.place(relx = 0.2, rely = 0.1)

# Volume buttons: Background sounds
noise_channel.set_volume(0.1)
up_sounds_button = tk.Button(text = "Up", command = lambda: noise_channel.set_volume(noise_channel.get_volume() + 0.1))
down_sounds_button = tk.Button(text = "Down", command = lambda: noise_channel.set_volume(noise_channel.get_volume() - 0.1))
up_sounds_button.place(relx = 0.1, rely = 0.2)
down_sounds_button.place(relx = 0.2, rely = 0.2)

def check_music_event():
    for event in pygame.event.get():
        if event.type == MUSIC_ENDED:
            window.after(30000, choose_music)
    
    window.after(1000, check_music_event)
MUSIC_ENDED = pygame.USEREVENT + 1
pygame.mixer.music.set_endevent(MUSIC_ENDED)
check_music_event()

def on_closing():
    pygame.mixer.music.stop()
    pygame.mixer.quit()      
    window.destroy()
window.protocol("WM_DELETE_WINDOW", on_closing)

# Execute the window
if __name__ == "__main__":
    window.mainloop()


In [ ]:
ambient = "Museum"
match ambient:
        case "Museum":
            sound = white_noises["Museum"]
        case "Resident services":
            sound = white_noises["Office"]
        case "Other": # Other settings with no background sound
            sound = ["Silence"]
sound


In [ ]:
pygame.mixer.music.pause()
noise_channel.pause()


TODO! 
1) Disegnare bottoni, cartello e slider
2) Aggiungere il banner con la data e l'ora 
3) Aggiungere lo slider e i bottoni
4) Provare l'app????
    

Aggiungere il bottone per cambiare la stagione. Colori dell'app che cambiano con la stagione? Alberi che cambiano colore? Sakura  e verde alternati per la primavera e arancio per l'autunno? Illustrazione stile libro per bambini per lo sfondo? O è troppo stimolante?
